# BirdCLEF 2026 — Submission Notebook (exp78 simple)

**Local model val AUC:** 0.8533  
**Experiment:** #78, topdb_40_optimal_preprocessing  
**Preprocessing:** n_fft=2048, hop_length=256, fmin=20, fmax=16000, n_mels=64, top_db=40, mel_norm=slaney, htk=True  
**Spectrogram shape:** (64, 626, 3)

**Inference:** official non-overlapping 5-second blocks only. No sliding windows, no overlap, no max aggregation.

In [1]:
import os
import numpy as np
import pandas as pd
import librosa
import tensorflow as tf
from tensorflow import keras
from keras import layers
import warnings
warnings.filterwarnings('ignore')

print('TensorFlow:', tf.__version__)
print('Librosa:', librosa.__version__)

2026-04-29 12:11:01.249536: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777464661.497536      16 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777464661.567729      16 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777464662.165798      16 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777464662.165859      16 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777464662.165863      16 computation_placer.cc:177] computation placer alr

TensorFlow: 2.19.0
Librosa: 0.11.0


In [2]:
# =======================================================
# PATHS
# =======================================================
WEIGHTS_PATH = '/kaggle/input/datasets/danielemalerba0302/birdclef2026-model/best_model_weights.weights.h5'
IMAGENET_WEIGHTS_PATH = '/kaggle/input/datasets/danielemalerba0302/birdclef2026-model/efficientnetb0_imagenet.weights.h5'
TEST_AUDIO_DIR = '/kaggle/input/competitions/birdclef-2026/test_soundscapes'
SAMPLE_SUB_PATH = '/kaggle/input/competitions/birdclef-2026/sample_submission.csv'
SUBMISSION_PATH = '/kaggle/working/submission.csv'

# =======================================================
# AUDIO PARAMETERS — EXACTLY matching local experiment #78
# experiment: topdb_40_optimal_preprocessing
# val_auc = 0.8533
# =======================================================
SAMPLE_RATE = 32000
DURATION = 5.0
N_MELS = 64
N_FFT = 2048
HOP_LENGTH = 256
FMIN = 20
FMAX = 16000
TOP_DB = 40.0
MEL_NORM = "slaney"
USE_HTK = True

TARGET_HEIGHT = N_MELS
TARGET_WIDTH = 626

# =======================================================
# MODEL ARCHITECTURE
# =======================================================
N_CLASSES = 234
DENSE_UNITS = 256
DROPOUT_RATE = 0.4

print(f'Spectrogram shape: ({TARGET_HEIGHT}, {TARGET_WIDTH}, 3)')
print(f'n_fft={N_FFT}, hop_length={HOP_LENGTH}, fmin={FMIN}, fmax={FMAX}')
print(f'mel_norm={MEL_NORM}, top_db={TOP_DB}, htk={USE_HTK}')

Spectrogram shape: (64, 626, 3)
Window: 5.0s, Step: 1.25s
mel_norm=slaney, top_db=60.0, htk=True


In [3]:
sample_sub = pd.read_csv(SAMPLE_SUB_PATH)
SPECIES_LIST = list(sample_sub.columns[1:])
print(f'Number of species: {len(SPECIES_LIST)}')

Number of species: 234


In [4]:
# Rebuild architecture and load weights
print('Rebuilding model architecture...')

base_model = keras.applications.EfficientNetB0(
    include_top=False,
    weights=None,
    input_shape=(TARGET_HEIGHT, TARGET_WIDTH, 3)
)
base_model.load_weights(IMAGENET_WEIGHTS_PATH)
base_model.trainable = False

inputs = keras.Input(shape=(TARGET_HEIGHT, TARGET_WIDTH, 3))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)
x = layers.Dense(DENSE_UNITS, activation='relu')(x)
x = layers.Dropout(DROPOUT_RATE)(x)
outputs = layers.Dense(N_CLASSES, activation='sigmoid')(x)
model = keras.Model(inputs, outputs)

model.load_weights(WEIGHTS_PATH)

print('Model loaded!')
print(f'  Input:  {model.input_shape}')
print(f'  Output: {model.output_shape}')

Rebuilding model architecture...


2026-04-29 12:11:31.542240: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Model loaded!
  Input:  (None, 64, 626, 3)
  Output: (None, 234)


In [5]:
def audio_to_spectrogram(audio_array, sr=SAMPLE_RATE):
    """
    Match experiment_template.py > make_melspec() for experiment #78.
    top_db=40, fmax=16000, n_mels=64, hop=256, mel_norm='slaney', htk=True.
    """
    mel = librosa.feature.melspectrogram(
        y=audio_array, sr=sr,
        n_mels=N_MELS,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        fmin=FMIN,
        fmax=FMAX,
        norm=MEL_NORM,
        htk=USE_HTK
    )

    mel = np.nan_to_num(mel, nan=0.0, posinf=0.0, neginf=0.0)
    mel_db = librosa.power_to_db(mel, ref=np.max, top_db=TOP_DB)
    mel_db = np.nan_to_num(mel_db, nan=-TOP_DB, posinf=0.0, neginf=-TOP_DB)
    mel_norm = (mel_db + TOP_DB) / TOP_DB
    mel_norm = np.clip(mel_norm, 0.0, 1.0)

    spec = np.stack([mel_norm, mel_norm, mel_norm], axis=-1).astype(np.float32)

    if spec.shape[1] < TARGET_WIDTH:
        spec = np.pad(spec, ((0, 0), (0, TARGET_WIDTH - spec.shape[1]), (0, 0)), mode='constant')
    elif spec.shape[1] > TARGET_WIDTH:
        spec = spec[:, :TARGET_WIDTH, :]

    return spec

# Sanity check: generated spectrogram must match model input exactly.
dummy = np.zeros(int(DURATION * SAMPLE_RATE), dtype=np.float32)
test_spec = audio_to_spectrogram(dummy)
expected_shape = tuple(model.input_shape[1:])
print(f'Test spectrogram shape: {test_spec.shape}')
print(f'Model input shape:       {expected_shape}')
assert test_spec.shape == expected_shape, f'Spectrogram/model shape mismatch: {test_spec.shape} != {expected_shape}'
print('Preprocessing shape OK!')

Test spectrogram shape: (64, 626, 3)
Preprocessing shape OK!


In [6]:
def process_audio_file_official_blocks(audio_path):
    """
    Load one test soundscape and split it into official non-overlapping 5-second blocks:
    0-5, 5-10, 10-15, ...

    Each block is predicted once and mapped to row_id = filename_endsecond.
    No sliding windows, no overlaps, no max aggregation.
    """
    filename = os.path.splitext(os.path.basename(audio_path))[0]
    results = []

    try:
        y, sr = librosa.load(audio_path, sr=SAMPLE_RATE, mono=True)

        block_samples = int(DURATION * SAMPLE_RATE)
        n_blocks = len(y) // block_samples

        for block_idx in range(n_blocks):
            start = block_idx * block_samples
            end = start + block_samples
            block = y[start:end]

            spec = audio_to_spectrogram(block, sr=sr)
            row_id = f'{filename}_{(block_idx + 1) * int(DURATION)}'
            results.append((row_id, spec))

    except Exception as e:
        print(f'  ERROR {filename}: {e}')

    return results

print('process_audio_file_official_blocks defined')

process_audio_file_sliding defined


In [7]:
if os.path.exists(TEST_AUDIO_DIR):
    audio_files = sorted([f for f in os.listdir(TEST_AUDIO_DIR) if f.endswith('.ogg')])
    print(f'Found {len(audio_files)} test files')
else:
    audio_files = []
    print('WARNING: test dir not found (normal in manual mode)')

predictions_by_row_id = {}
BATCH_SIZE = 32
batch_specs = []
batch_row_ids = []

def flush_batch():
    global batch_specs, batch_row_ids
    if not batch_specs:
        return
    X = np.array(batch_specs, dtype=np.float32)
    preds = model.predict(X, verbose=0)
    for row_id, pred in zip(batch_row_ids, preds):
        if row_id in predictions_by_row_id:
            raise ValueError(f'Duplicate prediction for row_id: {row_id}')
        predictions_by_row_id[row_id] = pred
    batch_specs = []
    batch_row_ids = []

for file_idx, filename in enumerate(audio_files):
    if file_idx % 10 == 0:
        print(f'  {file_idx+1}/{len(audio_files)}: {filename}')

    audio_path = os.path.join(TEST_AUDIO_DIR, filename)
    blocks = process_audio_file_official_blocks(audio_path)

    for row_id, spec in blocks:
        batch_row_ids.append(row_id)
        batch_specs.append(spec)
        if len(batch_specs) >= BATCH_SIZE:
            flush_batch()

flush_batch()
print(f'Inference complete! Predicted row_ids: {len(predictions_by_row_id)}')

Found 0 test files
Inference complete! Unique row_ids: 0


In [8]:
# Build submission in exactly the same row_id order as sample_submission.
expected_row_ids = list(sample_sub['row_id'])

if len(predictions_by_row_id) == 0:
    print('No predictions (normal in manual mode)')
    submission_df = pd.DataFrame(columns=list(sample_sub.columns))
else:
    predicted_row_ids = set(predictions_by_row_id.keys())
    expected_set = set(expected_row_ids)

    missing = sorted(expected_set - predicted_row_ids)
    extra = sorted(predicted_row_ids - expected_set)
    assert not missing, f'Missing {len(missing)} row_ids, first few: {missing[:5]}'
    assert not extra, f'Extra {len(extra)} row_ids, first few: {extra[:5]}'

    predictions_array = np.array([predictions_by_row_id[row_id] for row_id in expected_row_ids], dtype=np.float32)
    submission_df = pd.DataFrame(predictions_array, columns=SPECIES_LIST)
    submission_df.insert(0, 'row_id', expected_row_ids)

submission_df.to_csv(SUBMISSION_PATH, index=False)
print(f'Submission saved: {SUBMISSION_PATH}')
print(f'Shape: {submission_df.shape}')

No predictions (normal in manual mode)
Submission saved: /kaggle/working/submission.csv
Shape: (0, 235)


In [9]:
check_df = pd.read_csv(SUBMISSION_PATH)
sample_check = pd.read_csv(SAMPLE_SUB_PATH)

print('=== VERIFICATION ===')
print(f'Rows: {len(check_df)}')
print(f'Columns: {len(check_df.columns)} (expected 235)')
print(f'First col: {check_df.columns[0]}')

columns_match = list(check_df.columns) == list(sample_check.columns)
print(f'Columns exactly match sample_submission: {columns_match}')
assert columns_match, 'Submission columns do not exactly match sample_submission columns'

row_ids_match = list(check_df['row_id']) == list(sample_check['row_id'])
print(f'row_ids exactly match sample_submission: {row_ids_match}')
assert row_ids_match, 'Submission row_ids do not exactly match sample_submission row_ids'

if len(check_df) > 0:
    pred_values = check_df.iloc[:, 1:].values
    print(f'Min: {pred_values.min():.4f}')
    print(f'Max: {pred_values.max():.4f}')
    print(f'Has NaN: {np.isnan(pred_values).any()}')
    assert not np.isnan(pred_values).any(), 'Submission contains NaN values'
    assert pred_values.min() >= 0.0, 'Submission contains values below 0'
    assert pred_values.max() <= 1.0, 'Submission contains values above 1'
    print('Submission ready!')
else:
    print('No rows — normal in manual mode. Commit to get real predictions.')

=== VERIFICATION ===
Rows: 0
Columns: 235 (expected 235)
First col: row_id
No rows — normal in manual mode. Commit to get real predictions.
